#### 감정 분석 자연어 처리
1. 데이터 폴더 안에 ratings_train.txt 파일을 로드
2. 데이터를 상위 500개 데이터만 추출
3. 리뷰 데이터와 감정 데이터로 나눠준다
4. 리뷰 데이터를 토큰화(komoran 함수 이용)
5. Word2Vec 데이터의 학습
    - window -> 3
    - epochs -> 5
    - min_count -> 5
    - sg -> 1
    - seed -> 42
6. 벡터화(Word2Vec, 단위 벡터의 평균)
7. 분류 모델 ( SVC, Logistic )
8. train, test 를 이용하여 2개의 모델 중 성능이 높은 모델이 무엇인가?
9. 단위 벡터의 평균의 성능과 단위 벡터 + 중요도 평균의 성능의 차이를 확인

In [ ]:
import numpy as np 
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from konlpy.tag import Komoran

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../data/ratings_train.txt", sep='\t')

In [3]:
df_data = df.head(500).copy()

In [4]:
df_data.head(3)

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0


In [7]:
# 토큰화 함수 생성 
# komoran 사용 (konlpy 설치가 되어있는 경우)
# 설치가 되어있지 않은 경우에는 split()을 이용하여 토큰화 
def build_tokenize():
    try:
        from konlpy.tag import Komoran
        komoran = Komoran()
        allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'SL', 'MAG']
        def tokenize(text):
            tokens = []
            for word, pos in komoran.pos(text):
                if pos in allow_pos:
                    tokens.append(word)
            return tokens
        return tokenize
    except Exception as e:
        print("Komoran 사용 불가 : ", e)
        return lambda x: x.split()

In [5]:
X = df_data['document']
Y = df_data['label']

In [6]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    train_size=0.8,
    random_state=42,
    stratify=Y
)

In [8]:
tokenize = build_tokenize()

In [14]:
tfidf_vec = TfidfVectorizer(
    tokenizer= tokenize,
    lowercase= False,
).fit(X_train)
tfidf_vec

c:\Users\johnh\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,False
,preprocessor,None
,tokenizer,<function bui...001F6B9FE9260>
,analyzer,'word'
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"


In [ ]:
# 데이터의 개수가 적당히 많은 수준인 경우 Train, Test 로 데이터 분할
# Train 데이터를 이용해서 학습을 하고, Test 데이터를 이용해서 검증하는 함수
svc = SVC(random_state=42)

def run_model(X, Y, test_size=0.2):
    X_train, X_test, Y_train, Y_test = train_test_split(
        X, Y, test_size= test_size, random_state=42, stratify=Y
    )
    # 모델에 학습
    svc.fit(X_train, Y_train)
    # 학습된 모델에 예측값 생성
    y_pred = svc.predict(X_test)
    print("정확도 : ", accuracy_score(y_pred, Y_test))
    print("분류 레포트 : ") 
    print(classification_report(y_pred, Y_test))


In [16]:
# 학습된 모델에 예측의 값을 반환하는 함수
# 세번째 매개변수(vec_type)을 생성 -> 기본값음 'mean'
# 'tfidf' 입력이 들어온다면 벡터화 작업은 w2v + tfidf 융합한 벡터화
def predict_sentence_list(sentences, model, vec_type = 'mean'):
    # sentences : 문장들의 리스트
    # 문장들을 토큰화 -> 임베딩
    X_test = []
    for sent in sentences:
        # token() 함수를 호출하여 토큰화
        tokens = tokenize(sent)
        # 토큰화 된 문장을 sent_embed_mean 함수에 입력하여 호출 (단위 벡터의 평균)
        if vec_type == 'mean':    
            vec = sent_embed_mean(tokens)
        elif vec_type == 'tfidf':
            vec = sent_embed_tfidf(tokens)
        X_test.append(vec)
    
    preds = model.predict(X_test)
    result = []
    for sent, pred in zip(sentences, preds):
        label = '긍정' if pred == 1 else '부정'
        result.append([sent,label])
    return result


In [20]:
idf = dict(
    zip(
        # get_features_names_out() -> Tfidf 에서 사용된 단어들의 목록
        tfidf_vec.get_feature_names_out(),
        # idf_ : 중요도 
        tfidf_vec.idf_
    )
)

In [ ]:
# Word2Vec을 이용하여 학습(Skip-gram 방식)
w2v = Word2Vec(
    sentences= tokens, 
    vector_size= 100, 
    window = 5, 
    min_count= 1, 
    sg = 1, 
    epochs= 100, 
    seed = 42, 
    workers=2
)

In [21]:
def sent_embed_tfidf(tokens):
    vecs = []
    weight = []
    for word in tokens:
        # tokens에 각각의 단어가 Word2Vec과 TF-IDF에 존재한다면
        if word in wv.key_to_index and word in idf:
            # vecs -> 단위벡터와 중요도를 곱한 값을 vecs 추가
            vecs.append(wv[word]*idf[word])
            # weight -> 중요도 데이터를 추가
            weight.append(idf[word])
        # vecs에 데이터가 존재하지 않는다면 -> tokens 안에 단어는 존재하지만 Word2Vec이나
        # TF-IDF에 단어가 존재하지 않을때
    if not vecs:
        # 희소 행렬 되돌려준다. 0행렬
        result = np.zeros(wv.vectors_size)
    else:
        result = np.sum(vecs, axis=0) / (np.sum(weight)+ 1e-9)
    return result

In [22]:
# 문장은 임베딩하는 함수를 생성 -> 벡터화
# 단위 벡터의 평균을 구하는 함수
def sent_embed_mean(tokens):
    vecs = []
    for word in tokens:
        if word in wv.index_to_key:
            vecs.append(wv[word])
    result = np.mean(vecs, axis=0) if vecs else np.zeros(wv.vector_size)
    return result
